In [2]:
!pip install imageio-ffmpeg

   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
    --------------------------------------- 0.5/31.2 MB 10.0 MB/s eta 0:00:04
   -- ------------------------------------- 2.3/31.2 MB 24.4 MB/s eta 0:00:02
   ------ --------------------------------- 4.8/31.2 MB 33.8 MB/s eta 0:00:01
   ---------- ----------------------------- 8.3/31.2 MB 44.2 MB/s eta 0:00:01
   --------------- ------------------------ 12.3/31.2 MB 81.8 MB/s eta 0:00:01
   ------------------- -------------------- 15.5/31.2 MB 73.1 MB/s eta 0:00:01
   ------------------------ --------------- 18.9/31.2 MB 81.8 MB/s eta 0:00:01
   --------------------------- ------------ 21.8/31.2 MB 73.1 MB/s eta 0:00:01
   --------------------------------- ------ 25.9/31.2 MB 81.8 MB/s eta 0:00:01
   ------------------------------------- -- 29.4/31.2 MB 81.8 MB/s eta 0:00:01
   ---------------------------------------  31.2/31.2 MB 81.8 MB/s eta 0:00:01
   ---------------------------------------  31.2/31.2 MB 81.8 MB/


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\manas\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install imageio-ffmpeg

  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl (31.2 MB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Cell 1 — scan and print your ACTUAL folder tree
from pathlib import Path

BASE_DIR = Path(r"C:\CUNY\SPRING 2026\Speech and Audio Learning\Final_Project_SpeechLanguageImpairment")

print("BASE_DIR exists:", BASE_DIR.exists())
print("\nFull tree:\n")

for p in sorted(BASE_DIR.rglob("*")):
    depth = len(p.relative_to(BASE_DIR).parts)
    indent = "  " * depth
    print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")

BASE_DIR exists: True

Full tree:

  ENNI.zip
  ENNI_audio/
    SLI/
      A/
        413.mp3
        444.mp3
        479.mp3
        529.mp3
        568.mp3
        570.mp3
        572.mp3
        574.mp3
        607.mp3
        609.mp3
        617.mp3
        625.mp3
        678.mp3
        717.mp3
        721.mp3
        725.mp3
        729.mp3
        733.mp3
        777.mp3
        817.mp3
        821.mp3
        825.mp3
        829.mp3
        871.mp3
        878.mp3
        880.mp3
        922.mp3
        926.mp3
        973.mp3
      B/
        427.mp3
        480.mp3
        527.mp3
        531.mp3
        535.mp3
        567.mp3
        569.mp3
        576.mp3
        611.mp3
        667.mp3
        673.mp3
        679.mp3
        680.mp3
        723.mp3
        727.mp3
        731.mp3
        735.mp3
        739.mp3
        774.mp3
        778.mp3
        819.mp3
        823.mp3
        827.mp3
        831.mp3
        870.mp3
        872.mp3
        874.mp3
        875.mp3
 

In [2]:
import os
import re
import sys
from pathlib import Path

# ── Fix ffmpeg — add to PATH so pydub subprocess can find it ─────────────────
import imageio_ffmpeg
_ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
_ffmpeg_dir = str(Path(_ffmpeg_exe).parent)
os.environ["PATH"] = _ffmpeg_dir + os.pathsep + os.environ["PATH"]

from pydub import AudioSegment
AudioSegment.converter = _ffmpeg_exe
AudioSegment.ffprobe   = str(Path(_ffmpeg_exe).parent / "ffprobe.exe")

# Verify pydub can actually call it
import subprocess
result = subprocess.run([_ffmpeg_exe, "-version"], capture_output=True, text=True)
print("ffmpeg version check:", result.stdout.split("\n")[0])

import librosa
import soundfile as sf
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = Path(r"C:\CUNY\SPRING 2026\Speech and Audio Learning\Final_Project_SpeechLanguageImpairment")
AUDIO_DIR       = BASE_DIR / "ENNI_audio"
TRANSCRIPT_DIR  = BASE_DIR / "ENNI_transcripts"
OUTPUT_DIR            = BASE_DIR / "CHI_extracted"
OUTPUT_AUDIO_DIR      = OUTPUT_DIR  / "audio"
OUTPUT_TRANSCRIPT_DIR = OUTPUT_DIR  / "transcripts"

GROUPS  = ["SLI", "TD"]
SUBSETS = ["A", "B"]
SR      = 16000

# ── Helpers ───────────────────────────────────────────────────────────────────
def parse_chi_segments(cha_path):
    segments = []
    with open(cha_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("*CHI:"):
                match = re.search(r"(\d+)_(\d+)", line)
                if match:
                    start_ms = int(match.group(1))
                    end_ms   = int(match.group(2))
                    text     = re.sub(r"\d+_\d+", "", line[5:])
                    text     = re.sub(r"[\[\]<>%].*", "", text).strip(" .\n")
                    if end_ms - start_ms > 300:
                        segments.append({
                            "start": start_ms / 1000,
                            "end":   end_ms   / 1000,
                            "text":  text or f"{start_ms}_{end_ms}"
                        })
    return segments

def extract_chi_lines(cha_path):
    header_lines, chi_lines = [], []
    with open(cha_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("@"):
            header_lines.append(line)
        elif line.startswith("*CHI:"):
            chi_lines.append(line)
            j = i + 1
            while j < len(lines) and lines[j].startswith("%"):
                chi_lines.append(lines[j])
                j += 1
    return header_lines + chi_lines

def slice_chi_audio(mp3_path, segments, out_dir, stem):
    # Use ffmpeg directly via subprocess to convert mp3 → wav first,
    # then slice with librosa — avoids pydub subprocess issues on Windows
    full_wav = out_dir / f"_full_{stem}.wav"
    subprocess.run([
        _ffmpeg_exe, "-y", "-i", str(mp3_path),
        "-ar", str(SR), "-ac", "1",
        str(full_wav)
    ], capture_output=True, check=True)

    y_full, _ = librosa.load(str(full_wav), sr=SR, mono=True)
    full_wav.unlink()

    saved = []
    for idx, seg in enumerate(segments):
        start_sample = int(seg["start"] * SR)
        end_sample   = int(seg["end"]   * SR)
        chunk        = y_full[start_sample:end_sample]

        chunk, _ = librosa.effects.trim(chunk, top_db=20)
        chunk     = librosa.util.normalize(chunk)

        out_path = out_dir / f"{stem}_chi_{idx:03d}.wav"
        sf.write(str(out_path), chunk, SR)
        saved.append(out_path)
    return saved

# ── Main ──────────────────────────────────────────────────────────────────────
def process_dataset():
    stats = {"processed": 0, "segments_total": 0, "skipped": 0}

    for group in GROUPS:
        for subset in SUBSETS:
            audio_src      = AUDIO_DIR      / group / subset
            transcript_src = TRANSCRIPT_DIR / group / subset
            audio_out      = OUTPUT_AUDIO_DIR      / group / subset
            transcript_out = OUTPUT_TRANSCRIPT_DIR / group / subset

            audio_out.mkdir(parents=True, exist_ok=True)
            transcript_out.mkdir(parents=True, exist_ok=True)

            if not audio_src.exists() or not transcript_src.exists():
                print(f"  [SKIP] {group}/{subset} — folder not found")
                stats["skipped"] += 1
                continue

            mp3_files = sorted(audio_src.glob("*.mp3"))
            cha_files = {f.stem: f for f in transcript_src.glob("*.cha")}
            print(f"\n── {group}/{subset}  ({len(mp3_files)} audio files) ──")

            for mp3_path in mp3_files:
                stem = mp3_path.stem
                if stem not in cha_files:
                    print(f"  [WARN] No .cha for {stem}, skipping")
                    stats["skipped"] += 1
                    continue

                segments = parse_chi_segments(cha_files[stem])
                if not segments:
                    print(f"  [WARN] No CHI timestamps in {stem}.cha")
                    stats["skipped"] += 1
                    continue

                try:
                    saved = slice_chi_audio(mp3_path, segments, audio_out, stem)
                except Exception as e:
                    print(f"  [ERR]  {stem} — {e}")
                    stats["skipped"] += 1
                    continue

                chi_lines = extract_chi_lines(cha_files[stem])
                out_cha   = transcript_out / f"{stem}_chi.cha"
                with open(out_cha, "w", encoding="utf-8") as f:
                    f.writelines(chi_lines)

                print(f"  [OK]  {stem}  →  {len(saved)} segments + transcript")
                stats["processed"]      += 1
                stats["segments_total"] += len(saved)

    print("\n" + "="*52)
    print(f"  Files processed   : {stats['processed']}")
    print(f"  CHI audio segments: {stats['segments_total']}")
    print(f"  Skipped           : {stats['skipped']}")
    print(f"  Output            : {OUTPUT_DIR.resolve()}")
    print("="*52)

process_dataset()

c:\Users\manas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


ffmpeg version check: ffmpeg version 7.1-essentials_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers

── SLI/A  (29 audio files) ──
  [OK]  413  →  58 segments + transcript
  [OK]  444  →  68 segments + transcript
  [OK]  479  →  69 segments + transcript
  [OK]  529  →  77 segments + transcript
  [OK]  568  →  57 segments + transcript
  [OK]  570  →  71 segments + transcript
  [OK]  572  →  165 segments + transcript
  [OK]  574  →  83 segments + transcript
  [OK]  607  →  88 segments + transcript
  [OK]  609  →  133 segments + transcript
  [OK]  617  →  83 segments + transcript
  [OK]  625  →  73 segments + transcript
  [OK]  678  →  74 segments + transcript
  [OK]  717  →  75 segments + transcript
  [OK]  721  →  82 segments + transcript
  [OK]  725  →  201 segments + transcript
  [OK]  729  →  134 segments + transcript
  [OK]  733  →  66 segments + transcript
  [OK]  777  →  58 segments + transcript
  [OK]  817  →  69 segments + transcript
  [OK]  821  →  71 segments 

In [3]:
import os
import re
import subprocess
from pathlib import Path
import imageio_ffmpeg
import librosa
import soundfile as sf
import numpy as np

# ── Install noisereduce if missing ────────────────────────────────────────────
try:
    import noisereduce as nr
except ImportError:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "noisereduce"], check=True)
    import noisereduce as nr

# ── ffmpeg setup ──────────────────────────────────────────────────────────────
_ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
os.environ["PATH"] = str(Path(_ffmpeg_exe).parent) + os.pathsep + os.environ["PATH"]

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = Path(r"C:\CUNY\SPRING 2026\Speech and Audio Learning\Final_Project_SpeechLanguageImpairment")
AUDIO_DIR       = BASE_DIR / "ENNI_audio"
TRANSCRIPT_DIR  = BASE_DIR / "ENNI_transcripts"
OUTPUT_DIR      = BASE_DIR / "CHI_extracted"

UTTERANCE_DIR   = OUTPUT_DIR / "utterances"          # raw CHI clips
CLEANED_UTT_DIR = OUTPUT_DIR / "utterances_cleaned"  # noise-reduced + trimmed
MERGED_DIR      = OUTPUT_DIR / "merged"              # raw merged
CLEANED_MRG_DIR = OUTPUT_DIR / "merged_cleaned"      # noise-reduced only (no trim)
TRANSCRIPT_OUT  = OUTPUT_DIR / "transcripts"

GROUPS  = ["SLI", "TD"]
SUBSETS = ["A", "B"]
SR      = 16000

# ── Audio cleaning ────────────────────────────────────────────────────────────
def clean_audio_utterance(y, sr):
    """
    For individual utterance clips:
      1. Noise reduction  — n_jobs=1 to prevent memory exhaustion
      2. Silence trim     — safe here, silence has no acoustic feature value
      3. Normalize        — peak normalize to [-1, 1]
    """
    try:
        noise_sample = y[:int(sr * 0.3)] if len(y) > int(sr * 0.3) else y
        y = nr.reduce_noise(
            y             = y,
            sr            = sr,
            y_noise       = noise_sample,
            prop_decrease = 0.75,
            stationary    = False,
            n_jobs        = 1        # ← was -1, caused RAM exhaustion
        )
    except Exception:
        pass

    y, _ = librosa.effects.trim(y, top_db=20)
    return librosa.util.normalize(y) if np.max(np.abs(y)) > 0 else y


def clean_audio_merged(y, sr):
    """
    For merged per-child files:
      1. Noise reduction  — chunked processing for long files to avoid OOM
      2. NO silence trim  — silence gaps are clinically meaningful
      3. Normalize        — peak normalize to [-1, 1]
    """
    # Merged files can be 5–8 mins long — process in 30s chunks to stay in RAM
    chunk_len  = sr * 30
    y_out      = np.zeros_like(y)

    if len(y) <= chunk_len:
        # Short enough — process whole file at once
        try:
            noise_sample = y[:int(sr * 0.3)] if len(y) > int(sr * 0.3) else y
            y_out = nr.reduce_noise(
                y             = y,
                sr            = sr,
                y_noise       = noise_sample,
                prop_decrease = 0.75,
                stationary    = False,
                n_jobs        = 1
            )
        except Exception:
            y_out = y
    else:
        # Long file — estimate noise from very start of session, then chunk
        noise_sample = y[:int(sr * 0.3)]
        n_chunks     = int(np.ceil(len(y) / chunk_len))

        for i in range(n_chunks):
            s   = i * chunk_len
            e   = min(s + chunk_len, len(y))
            seg = y[s:e]
            try:
                seg_clean = nr.reduce_noise(
                    y             = seg,
                    sr            = sr,
                    y_noise       = noise_sample,
                    prop_decrease = 0.75,
                    stationary    = False,
                    n_jobs        = 1
                )
                y_out[s:e] = seg_clean
            except Exception:
                y_out[s:e] = seg       # fallback: keep original chunk

    return librosa.util.normalize(y_out) if np.max(np.abs(y_out)) > 0 else y_out


# ── Parse .cha → CHI segments ─────────────────────────────────────────────────
def parse_chi_segments(cha_path):
    segments = []
    with open(cha_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("*CHI:"):
                match = re.search(r"(\d+)_(\d+)", line)
                if match:
                    start_ms = int(match.group(1))
                    end_ms   = int(match.group(2))
                    text     = re.sub(r"\d+_\d+", "", line[5:])
                    text     = re.sub(r"[\[\]<>%].*", "", text).strip(" .\n")
                    if end_ms - start_ms > 300:
                        segments.append({
                            "start": start_ms / 1000,
                            "end":   end_ms   / 1000,
                            "text":  text or f"{start_ms}_{end_ms}"
                        })
    return segments


# ── Parse .cha → CHI-only transcript ─────────────────────────────────────────
def extract_chi_lines(cha_path):
    header_lines, chi_lines = [], []
    with open(cha_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("@"):
            header_lines.append(line)
        elif line.startswith("*CHI:"):
            chi_lines.append(line)
            j = i + 1
            while j < len(lines) and lines[j].startswith("%"):
                chi_lines.append(lines[j])
                j += 1
    return header_lines + chi_lines


# ── Convert full mp3 → wav ────────────────────────────────────────────────────
def load_full_audio(mp3_path, tmp_dir):
    full_wav = tmp_dir / f"_full_{mp3_path.stem}.wav"
    subprocess.run([
        _ffmpeg_exe, "-y", "-i", str(mp3_path),
        "-ar", str(SR), "-ac", "1", str(full_wav)
    ], capture_output=True, check=True)
    y, _ = librosa.load(str(full_wav), sr=SR, mono=True)
    full_wav.unlink()
    return y


# ── Save utterances (raw + cleaned) ──────────────────────────────────────────
def save_utterances(y_full, segments, raw_dir, clean_dir, stem):
    saved_raw, saved_clean = [], []

    for idx, seg in enumerate(segments):
        s     = int(seg["start"] * SR)
        e     = int(seg["end"]   * SR)
        chunk = y_full[s:e]

        if len(chunk) < SR * 0.1:
            continue

        # raw: trim + normalize only
        chunk_raw, _ = librosa.effects.trim(chunk.copy(), top_db=20)
        chunk_raw    = librosa.util.normalize(chunk_raw)
        raw_path     = raw_dir / f"{stem}_chi_{idx:03d}.wav"
        sf.write(str(raw_path), chunk_raw, SR)
        saved_raw.append(raw_path)

        # cleaned: noise reduce + trim + normalize
        chunk_clean = clean_audio_utterance(chunk.copy(), SR)
        clean_path  = clean_dir / f"{stem}_chi_{idx:03d}.wav"
        sf.write(str(clean_path), chunk_clean, SR)
        saved_clean.append(clean_path)

    return saved_raw, saved_clean


# ── Save merged (raw + cleaned) ───────────────────────────────────────────────
def save_merged(y_full, segments, raw_dir, clean_dir, stem):
    if not segments:
        return None, None

    # Zero out examiner turns, keep CHI turns at original position
    merged = np.zeros(len(y_full), dtype=np.float32)
    for seg in segments:
        s = int(seg["start"] * SR)
        e = int(seg["end"]   * SR)
        merged[s:e] = y_full[s:e]

    # Crop from first to last CHI utterance (removes pre/post session silence)
    first_s = int(segments[0]["start"] * SR)
    last_e  = int(segments[-1]["end"]  * SR)
    merged  = merged[first_s:last_e]

    # raw merged — normalize only
    raw_path = raw_dir / f"{stem}_chi_merged.wav"
    sf.write(str(raw_path), librosa.util.normalize(merged.copy()), SR)

    # cleaned merged — noise reduce + normalize, NO trim
    clean_path = clean_dir / f"{stem}_chi_merged.wav"
    sf.write(str(clean_path), clean_audio_merged(merged.copy(), SR), SR)

    return raw_path, clean_path


# ── Main ──────────────────────────────────────────────────────────────────────
def process_dataset():
    stats = {"processed": 0, "utterances": 0, "skipped": 0}

    for group in GROUPS:
        for subset in SUBSETS:
            audio_src      = AUDIO_DIR      / group / subset
            transcript_src = TRANSCRIPT_DIR / group / subset

            utt_raw   = UTTERANCE_DIR   / group / subset
            utt_clean = CLEANED_UTT_DIR / group / subset
            mrg_raw   = MERGED_DIR      / group / subset
            mrg_clean = CLEANED_MRG_DIR / group / subset
            trans_out = TRANSCRIPT_OUT  / group / subset

            for d in [utt_raw, utt_clean, mrg_raw, mrg_clean, trans_out]:
                d.mkdir(parents=True, exist_ok=True)

            if not audio_src.exists() or not transcript_src.exists():
                print(f"  [SKIP] {group}/{subset} — folder not found")
                stats["skipped"] += 1
                continue

            mp3_files = sorted(audio_src.glob("*.mp3"))
            cha_files = {f.stem: f for f in transcript_src.glob("*.cha")}
            print(f"\n── {group}/{subset}  ({len(mp3_files)} files) ──")

            for mp3_path in mp3_files:
                stem = mp3_path.stem
                if stem not in cha_files:
                    print(f"  [WARN] {stem} — no matching .cha")
                    stats["skipped"] += 1
                    continue

                segments = parse_chi_segments(cha_files[stem])
                if not segments:
                    print(f"  [WARN] {stem} — no CHI timestamps")
                    stats["skipped"] += 1
                    continue

                try:
                    y_full = load_full_audio(mp3_path, utt_raw)

                    raw_utts, clean_utts = save_utterances(
                        y_full, segments, utt_raw, utt_clean, stem
                    )
                    save_merged(y_full, segments, mrg_raw, mrg_clean, stem)

                    chi_lines = extract_chi_lines(cha_files[stem])
                    with open(trans_out / f"{stem}_chi.cha", "w", encoding="utf-8") as f:
                        f.writelines(chi_lines)

                    print(f"  [OK]  {stem}  →  {len(raw_utts)} utterances  (raw + cleaned)  +  merged  +  transcript")
                    stats["processed"]  += 1
                    stats["utterances"] += len(raw_utts)

                except Exception as e:
                    print(f"  [ERR] {stem} — {e}")
                    stats["skipped"] += 1

    print("\n" + "="*52)
    print(f"  Files processed   : {stats['processed']}")
    print(f"  Total utterances  : {stats['utterances']}")
    print(f"  Skipped           : {stats['skipped']}")
    print(f"  Output            : {OUTPUT_DIR.resolve()}")
    print("="*52)


process_dataset()


── SLI/A  (29 files) ──
  [OK]  413  →  58 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  444  →  68 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  479  →  69 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  529  →  77 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  568  →  57 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  570  →  71 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  572  →  165 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  574  →  83 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  607  →  88 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  609  →  133 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  617  →  83 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  625  →  73 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  678  →  74 utterances  (raw + cleaned)  +  merged  +  transcript
  [OK]  717  →  75 utter